# Import library and support function

In [34]:
import sys
sys.path.insert(0, "/home/jupyter-hanx/SVM_review/utils")
sys.path.insert(0, "/home/jupyter-hanx/SVM_review/")
from utils.DataLoader.dataset_BoT_IoT import BoT_IoT
from utils.DataLoader.dataset_CIC_DDoS_2019 import CIC_DDoS_2019
from utils.DataLoader.dataset_CIC_IDS_2017 import CICIDS2017
from utils.load_data_ids import load_data_ids

from sklearn import svm, base, ensemble, metrics, model_selection, preprocessing, tree, linear_model, datasets
from sklearn.preprocessing import KernelCenterer, StandardScaler, LabelEncoder, OneHotEncoder, QuantileTransformer, MinMaxScaler, Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (ConfusionMatrixDisplay, roc_auc_score, precision_score, average_precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error, roc_curve, auc, classification_report,auc,confusion_matrix,matthews_corrcoef)
from sklearn.datasets import make_blobs, make_multilabel_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier, LocalOutlierFactor
from sklearn.svm import SVC, NuSVC

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.utils import resample
from sklearn.impute import SimpleImputer


from lightgbm import LGBMClassifier
from pyod.models.lof import LOF
from pyod.models.iforest import IForest
from pyod.models.ocsvm import OCSVM

import plotly.express as px

import os
import numpy as np
import scipy as sp
import pandas as pd
import urllib.request
import shutil
import tarfile
import seaborn as sns
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

from scipy.sparse.linalg import cg

import tensorflow as tf
from tqdm.notebook import trange, tqdm

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import spsolve

import matplotlib.pyplot as plt
import seaborn as sns
import time
import logging
from typing import List, Tuple, Generator, Iterator
import random
from decimal import *

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 



## Support function

In [35]:
#@title Hàm đo chỉ số
# basic random seed
__CUSTOM_COLS=['MCC', 'ACC','TPR', 'FPR', 'F1', 'TPR macro','PPV macro','F1 macro',"AUC","Training time","Testing time"]
__DEFAULT_RANDOM_SEED = 42

def seedEverything(seed=__DEFAULT_RANDOM_SEED):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)
  np.random.seed(seed)

  # # tensorflow random seed 
  # import tensorflow as tf 
  # tf.random.set_seed(seed)
    
  # # torch random seed
  # import torch
  # torch.manual_seed(seed)
  # torch.cuda.manual_seed(seed)
  # torch.backends.cudnn.deterministic = True
  # torch.backends.cudnn.benchmark = False



def cfs_matrix(y_label, y_pred, labels):
  
    cfs_mt = np.full((len(labels), len(labels)), 0)

    for x,y in zip(y_label, y_pred):
        cfs_mt[x,y] += 1
    
    return cfs_mt

def calc_index(testdf, Label_name, export_fig):

  #     P
  #     0   1
  # T 0 TN FP
  #   1 FN TP


    cnf_matrix = confusion_matrix(testdf.label, testdf.y_pred, labels = np.unique(testdf.label))
  #   # plot_confusion_matrix("Model_cfs",cnf_matrix, target_names=Label_name,figsize = (20, 10), export_fig=export_fig)

    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix) 
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)

    FPR = FP/(FP+TN) *100.

    ACC = (TP+TN)/(TP+FP+FN+TN) *100.

    auc_func = -1
    if len(Label_name) == 2:
        false_positive_rate, true_positive_rate, thresholds = roc_curve(testdf.label,  testdf.y_pred)
        auc_func = auc(false_positive_rate, true_positive_rate)

    tpr_func = recall_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100. 
    ppv_func = precision_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100.
    f1_func = f1_score(testdf.label,testdf.y_pred,average='macro', zero_division = 0) *100.
    mcc_func = matthews_corrcoef(testdf.label,testdf.y_pred)

    ACC = sum(TP+TN)/(sum(TP+FP+FN+TN)) *100.

    FPR = sum(FP)/sum((FP+TN)) *100.

    
    print("TPR-macro: {:.4f}".format(tpr_func))
    print("FPR      : {:.4f}".format(FPR))
    print("PPV-macro: {:.4f}".format(ppv_func))
    print("F1-macro : {:.4f}".format(f1_func))
    print("MCC-func  : {:.4f}".format(mcc_func))
    print("AUC-func  : {:.4f}".format(auc_func))
    print("CFS MATRIX:\n",cnf_matrix)
    
    return mcc_func, ACC, tpr_func, FPR, ppv_func, f1_func, auc_func, cnf_matrix    

def Average(lst):
  return sum(lst) / len(lst)


def plot_confusion_matrix(name,cm,
                          target_names,
                          title='Confusion matrix',
                          figsize=(15,8),
                          cmap=None,
                          normalize=False,export_fig=None):

  import matplotlib.pyplot as plt
  import numpy as np
  import itertools

  accuracy = np.trace(cm) / np.sum(cm).astype('float')
  misclass = 1 - accuracy

  if cmap is None:
    cmap = plt.get_cmap('Blues')
  norm_cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]*100
  plt.figure(figsize=figsize)
  plt.imshow(norm_cm, interpolation='nearest', cmap=cmap)
  plt.colorbar()

  if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=45, fontsize='large')
    plt.yticks(tick_marks, target_names, fontsize='large')

  thresh = cm.max() / 1.5 if normalize else cm.max() / 2
  thresh= 50
  for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):  
    plt.text(j, i, "{:,}\n{:0.2f}%".format(cm[i, j], norm_cm[i, j]),
                horizontalalignment="center",
                verticalalignment="center",
                color="white" if norm_cm[i, j] > thresh else "black")

  plt.tight_layout()
  plt.ylabel('True label', fontsize='x-large')
  plt.xlabel('Predicted label', fontsize='x-large')

  if export_fig is not None:
    plt.savefig(os.path.join(export_fig,'cfs_matrix.png'), bbox_inches='tight')
  plt.show()


def Export_report(id_information=[], clf_report=None, cnf_matrix=None, custom_report=None, Label_name=[], path="", mode='w'):
  
  df_header = pd.DataFrame([["BEGIN TEST"],[""],id_information,[""]])
  df_row = pd.DataFrame([[""]])
  df_end = pd.DataFrame(["","END TEST",""])
  df_1 = df_2 = df_3 = pd.DataFrame()

  if clf_report is not None:
    df_tmp = pd.DataFrame(clf_report).transpose()
    df_tmp = pd.concat([df_tmp.columns.to_frame().T, df_tmp], ignore_index=False)
    df_tmp.reset_index(inplace=True)
    df_tmp = df_tmp.rename(columns = {'index':''})
    df_1 = pd.concat([df_tmp, df_row], ignore_index=True)
    df_1.set_axis([*range(df_1.shape[1])],axis = 1, inplace=True)
  
  if cnf_matrix is not None:
    df_2 = pd.DataFrame(cnf_matrix)
    df_2.insert(0,'',Label_name)
    df_tmp = pd.DataFrame([Label_name])
    df_2 = pd.concat([df_tmp,df_2,df_row], axis = 0, ignore_index=True)
    df_2.set_axis([*range(df_2.shape[1])],axis = 1, inplace=True)
  if custom_report is not None:
    df_3 = pd.concat([custom_report.columns.to_frame().T, custom_report], ignore_index=False)
    df_3.reset_index(inplace=True)
    df_3 = df_3.rename(columns = {'index':'Model'})
    df_3.set_axis([*range(df_3.shape[1])],axis = 1, inplace=True)

  
  pd.concat([df_header, df_1, df_2, df_3, df_end], ignore_index=True).to_csv(path, header=False, index=False, mode=mode)

TupleOrList = tuple([Tuple, List])
class CustomMerger(base.BaseEstimator, base.TransformerMixin):
  """Merge List of DataFrames"""

  def __init__(self):
    pass

  def fit(self, X: pd.DataFrame, y=None):
    return self

  def transform(self, X: pd.DataFrame, y='deprecated', copy=True):
    if isinstance(X, TupleOrList):
        return pd.concat(X, ignore_index=True).reset_index(drop=True)

    return X


def plot_2d_features(X_train, y_train):
    # Apply PCA for dimensionality reduction to 2 features
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X_train)

    # Create a scatter plot
    unique_labels = np.unique(y_train)
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_labels)))

    plt.figure(figsize=(8, 6))

    for label, color in zip(unique_labels, colors):
        plt.scatter(X_2d[y_train == label, 0], X_2d[y_train == label, 1], color=color, label=str(label))

    plt.title('2D Feature Plot')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.show()


def plot_tsne_2d(X_train, y_train, perplexity = 30):
    if X_train.shape[1] > 50:
        X_train, pca = apply_pca(X_train)
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(X_train)
    
    # Create a scatter plot
    # unique_labels = np.unique(y_train)
    # labels = [str(x) for x in unique_labels]
    fig = px.scatter(x=X_tsne[:, 0], y=X_tsne[:, 1],color=y_train)
    
    # fig.update_traces(marker_size=8)
    fig.update_layout(height=900)
    fig.update_layout(width=900)
    fig.show(renderer='iframe')


    # fig = px.scatter(data_frame  =X_tsne, x=0, y=1, color=y_train, labels=labels)
    # fig.show()

def plot_tsne_3d(X_train, y_train, perplexity = 30):
    # Apply t-SNE for dimensionality reduction to 2 features

    if X_train.shape[1] > 50:
        X_train, pca = apply_pca(X_train)
        
    tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(X_train)
    
    # Create a scatter plot

    fig = px.scatter_3d(x=X_tsne[:, 0], y=X_tsne[:, 1], z=X_tsne[:, 2],color=y_train)
    
    # fig.update_traces(marker_size=8)
    fig.update_layout(height=900)
    fig.update_layout(width=900)
    fig.show(renderer='iframe')


def remove_outliers_lof(X_data, y_data, contamination=0.05, random_seed=None):
    """
    Remove outliers from a dataset using Local Outlier Factor (LOF).

    Parameters:
    - X_data: numpy array, feature matrix
    - y_data: numpy array, label array
    - contamination: float, the proportion of outliers in the dataset
    - random_seed: int or None, seed for reproducibility

    Returns:
    - X_no_outliers: numpy array, feature matrix without outliers
    - y_no_outliers: numpy array, label array without outliers
    """

    unique_classes = np.unique(y_data)

    X_no_outliers = np.empty((0, X_data.shape[1]), dtype=X_data.dtype)
    y_no_outliers = np.empty(0, dtype=y_data.dtype)

    for label in unique_classes:
        # Select samples belonging to the current class
        # print(label)
        class_mask = (y_data == label)
        X_class = X_data[class_mask]
        if label == 0:
            X_no_outliers = np.vstack((X_no_outliers, X_class))
            y_no_outliers = np.concatenate((y_no_outliers, y_data[class_mask]))
        else:
            # Apply LOF to detect outliers
            lof = LocalOutlierFactor(contamination=contamination)
            outliers_mask = lof.fit_predict(X_class) == -1

            # Remove outliers from the current class
            X_no_outliers = np.vstack((X_no_outliers, X_class[~outliers_mask]))
            y_no_outliers = np.concatenate((y_no_outliers, y_data[class_mask][~outliers_mask]))

    return X_no_outliers, y_no_outliers


def calculate_mean_point(cluster_data):
    """
    Calculate the mean point of a cluster.

    Parameters:
    - cluster_data: A list of vectors (NumPy arrays) representing the data points in the cluster.

    Returns:
    - mean_point: The mean point of the cluster.
    """

    # Convert the list of vectors to a NumPy array for efficient computation
    cluster_array = np.array(cluster_data)

    # Calculate the mean along each dimension (axis=0)
    mean_point = np.mean(cluster_array, axis=0)

    return mean_point



def apply_pca(input_array):
    """
    Perform PCA on a NumPy array and reduce its dimensionality.

    Parameters:
    - input_array: The input NumPy array.
    - n_components: The number of components (dimensions) to reduce to.

    Returns:
    - reduced_array: The NumPy array with reduced dimensionality.
    """
    # if n_components >= input_array.shape[1]:
    #     raise ValueError("Number of components should be less than the input array's number of features.")

    # Create PCA instance
    pca = PCA()

    # Fit and transform the input array
    reduced_array = pca.fit_transform(input_array)
    return reduced_array, pca


def apply_lda(X,y):
    """
    Perform PCA on a NumPy array and reduce its dimensionality.

    Parameters:
    - input_array: The input NumPy array.
    - n_components: The number of components (dimensions) to reduce to.

    Returns:
    - reduced_array: The NumPy array with reduced dimensionality.
    """
    # if n_components > min(X.shape[1], len(np.unique(y)) -1):
    #     raise ValueError("Number of components should be less than the input array's number of features.")

    # Create LDA instance
    lda = LinearDiscriminantAnalysis()

    # Fit and transform the input array
    reduced_array = lda.fit_transform(X,y)

    return reduced_array, lda

def visualize_multiclass_dataset(df):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x="Fts_0", y="Fts_1", hue="Label", data=df, palette="Set1", s=50, edgecolor='w')
    plt.title("Multi-Class Dataset with Anomalies")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend(title="Label", loc='upper right')
    plt.show()





## Function to Load Dataset

In [36]:
__DEVICES = ['Danmini_Doorbell', 'Ecobee_Thermostat', 'Ennio_Doorbell', 'Philips_B120N10_Baby_Monitor', 'Provision_PT_737E_Security_Camera', 'Provision_PT_838_Security_Camera', 'Samsung_SNH_1011_N_Webcam', 'SimpleHome_XCS7_1002_WHT_Security_Camera', 'SimpleHome_XCS7_1003_WHT_Security_Camera']


def Import_dataset(data_dir, devices = __DEVICES, bot_type = 2, atk_type = "ALL"):
    
    bot_names = ['mirai','gafgyt']
    __atk_names = {'mirai': ['ack','scan','syn','udp','udpplain'],
                 'gafgyt': ['combo','junk','scan','tcp','udp']
                }
    atk_names = []
    
    if bot_type == 0:
      bot_names = ['mirai']
    elif bot_type == 1:
      bot_names = ['gafgyt']
    
    if atk_type == "ALL":
      for x in bot_names:
        atk_names = atk_names + __atk_names[x]
    else:
      atk_names = atk_type
    
    
    print("Load normal file:")
    
    df_nors = []
    for device_name in devices:
      device_id = __DEVICES.index(device_name) + 1
      file_path = os.path.join(data_dir,'{}.benign.csv'.format(device_id))
      print("Load file benign:", file_path)
      if os.path.exists(file_path) == False:
        print('File {} not found. Ignored'.format(file_path))
        continue
        
      normal = pd.read_csv(file_path)
      # dropping duplicate values 
      normal.drop_duplicates(keep=False, inplace=True) 

      normal['Names Atk'] = np.array(['Benign']*normal.shape[0])
      normal['Names Bot'] = np.array(['Benign']*normal.shape[0])
      normal['Devices'] = np.array([device_name]*normal.shape[0])
      df_nors.append(normal)
      
    # CustomMerger.transform(df_nors)
    df_nors = pd.concat(df_nors, ignore_index=True).reset_index(drop=True)
    df_nors['Label'] = np.array([0]*df_nors.shape[0])
    
    print("Normal shape:", df_nors.shape)
    
    print("Load abnormal file:")
    
    df_anos = []
    for device_name in devices:
      device_id = __DEVICES.index(device_name) + 1
      df_bots = []
      for bot in bot_names:
        df_atks = []
        for atk in __atk_names[bot]:
          if atk not in atk_names:
            continue
          file_path = os.path.join(data_dir,'{}.{}.{}.csv'.format(device_id,bot,atk))
          print("Load file benign:", file_path)
          if os.path.exists(file_path) == False:
            print("File {} not found. Ignore".format(file_path))
            continue
          atk_sample = pd.read_csv(file_path)
          atk_sample.drop_duplicates(keep=False, inplace=True) 
            
          atk_sample['Names Atk'] = np.array([atk]*atk_sample.shape[0])
          df_atks.append(atk_sample) 
                                   
        # CustomMerger.transform(df_atks)
        if df_atks:
          df_atks = pd.concat(df_atks, ignore_index=True).reset_index(drop=True)
          df_atks['Names Bot'] = np.array([bot]*df_atks.shape[0])
          df_bots.append(df_atks)
                                   
      # CustomMerger.transform(df_bots)
      if df_bots:
        df_bots = pd.concat(df_bots, ignore_index=True).reset_index(drop=True)
        df_bots['Devices'] = np.array([device_name]*df_bots.shape[0])
        df_anos.append(df_bots)  
                                   
    # CustomMerger.transform(df_anos)
    if df_anos:
      df_anos = pd.concat(df_anos, ignore_index=True).reset_index(drop=True)
      df_anos['Label'] = np.array([1]*df_anos.shape[0])
    print("Abnormal shape:", df_anos.shape)
    
    return df_nors, df_anos

def Preprocess_data(df):
  df.drop(columns=['Label','Names Bot','Devices'], inplace=True, errors='ignore')
  df.rename(columns={'Names Atk': __TARGET}, inplace=True)
  return df

def Get_sample_ratio_by_column(df, col, max_cnt):
  dfse = df[col].value_counts()
  ans = []
  for x in dfse.index:
    tmp = df[df[col] == x].sample(n = max_cnt, random_state=__SEED)
    ans.append(tmp)
  return pd.concat(ans, ignore_index=True).reset_index(drop=True) 



def Train_test_split_unsup(df, rate):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    df_normal[__TARGET] = df_normal[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
    df_attack[__TARGET] = df_attack[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
    
    df_train_nor, df_test_nor = train_test_split(df_normal, test_size=rate, random_state=__SEED)
    
    df_test = pd.concat([df_test_nor, df_attack], axis=0)
    
    
    return df_train_nor, df_test

def Train_test_split_one_atk_unsup(df, rate):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    df_train_nor, df_test_nor = train_test_split(df_normal, test_size=rate, random_state=__SEED)
        
    dfs = []
    for name in __LIST_ATK:
        df_tmp = df_attack[df_attack[__TARGET] == name]  
        df_tmp = pd.concat([df_test_nor, df_tmp], axis=0)
        df_tmp[__TARGET] = df_tmp[__TARGET].apply(lambda x: 0 if (x == 'BENIGN' or x == 'Normal') else 1)
        dfs.append(df_tmp)
    
    
    return df_train_nor, pd.Series(data=dfs,index=__LIST_ATK)
    
def Normal_attack_split(df):
    global __UNSUP, __LIST_ATK
    __UNSUP = "UNSUP"
    __LIST_ATK = df[__TARGET].unique()
    
    __LIST_ATK = __LIST_ATK[ (__LIST_ATK != "BENIGN") & (__LIST_ATK != "Normal") ]

    df_normal = df[df[__TARGET].isin(['BENIGN', 'Normal'])]
    df_attack = df[df[__TARGET].isin(__LIST_ATK)]
    
    return df_normal, df_attack

In [37]:
__Atks_profile = {
    "CIC_IDS_2017":{
        # 'Malware': ['Bot']
    },
    "CIC_DDoS_2019":{
    },
    "BoT_IoT":{
        'Malware': ['Theft'],
        'DoS': ['DoS'],
        'DDoS': ['DDoS'],
        'Scan': ['Reconnaissance']
    },
    
    "ToN_IoT":{
        'Malware': ['backdoor','ransomware'],
        'Scan': ['scanning'],
        'BruteForce': ['password'],
        'DDoS': ['ddos'],
        'WebAttack': ['xss','injection'],
        'DoS': ['dos']
    },
    "N_BaIoT":{ # not map
        'Scan': ['scan'],
        'DoS': ['udp','tcp','syn','ack','udpplain','combo','junk'],
    },
    "UNSW_NB15":{
        'Malware': ['Exploits','Shellcode','Backdoor','Worms'],
        'DoS': ['DoS','Fuzzers'],
        'Scan': ['Reconnaissance','Analysis'],
    },
    "CIC_IoT2023": {
        'BruteForce': ['DictionaryBruteForce'],
        'Scan': ['Recon-PingSweep', 'Recon-OSScan', 'VulnerabilityScan', 'Recon-PortScan','Recon-HostDiscovery'],
        'WebAttack': ['SqlInjection', 'CommandInjection', 'Backdoor_Malware', 'Uploading_Attack','XSS','BrowserHijacking'],
        'Mirai': ['Mirai-greip_flood', 'Mirai-greeth_flood', 'Mirai-udpplain'],
    	'DDoS':['DDoS-ICMP_Flood', 'DDoS-UDP_Flood', 'DDoS-TCP_Flood', 'DDoS-SYN_Flood', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SynonymousIP_Flood',  'DDoS-ICMP_Fragmentation', 'DDoS-ACK_Fragmentation', 'DDoS-UDP_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-SlowLoris'],
    	'DoS': ['DoS-UDP_Flood', 'DoS-TCP_Flood', 'DoS-SYN_Flood', 'DoS-HTTP_Flood'],
    	'Web': ['Backdoor_Malware', 'Uploading_Attack' 'CommandInjection', 'XSS', 'SqlInjection', 'BrowserHijacking'],
    	'Spoofing': ['MITM-ArpSpoofing', 'DNS_Spoofing']
    }

}

_category_map = {
    'CIC_IoT2023': {
      '0Normal':            "0Normal",
      # 'DDoS-ACK_Fragmentation':   "DDoS",
      'DDoS-UDP_Flood':           "DDoS/DoS",
      # 'DDoS-SlowLoris':           "DDoS",
      'DDoS-ICMP_Flood':          "DDoS/DoS",
      # 'DDoS-RSTFINFlood':         "DDoS/DoS",
      # 'DDoS-PSHACK_Flood':        "DDoS",
      # 'DDoS-HTTP_Flood':          "DDoS",
      # 'DDoS-UDP_Fragmentation':   "DDoS",
      # 'DDoS-ICMP_Fragmentation':  "DDoS/DoS",
      'DDoS-TCP_Flood':           "DDoS/DoS",
      # 'DDoS-SYN_Flood':           "DDoS",
      # 'DDoS-SynonymousIP_Flood':  "DDoS",
        
      # 'DictionaryBruteForce':     "BruteForce",
      'MITM-ArpSpoofing':         'Spoofing',
      
        # 'DNS_Spoofing':             'Spoofing',
      'DoS-TCP_Flood':            'DDoS/DoS',
      # 'DoS-HTTP_Flood':           'DoS',
      # 'DoS-SYN_Flood':            'DoS',
      'DoS-UDP_Flood':            'DDoS/DoS',
      # 'Recon-PingSweep':          'Scan',
      # 'Recon-OSScan':             'Scan',
      
        'VulnerabilityScan':        'Scan',
      
        # 'Recon-PortScan':           'Scan',
      # 'Recon-HostDiscovery':      'Scan',
      # 'SqlInjection':             'Web_1',
      # 'CommandInjection':         'Web_2',
     
        'Backdoor_Malware':         'Web',
      # 'Uploading_Attack':         'Web_4',
      # 'XSS':                      'Web_5',
      # 'BrowserHijacking':         'Web_6',
      # 'Mirai-greip_flood':        'Mirai',
      # 'Mirai-greeth_flood':       'Mirai',
      
        'Mirai-udpplain':           'Mirai'
    },
    # 'N_BaIoT': {
    #   '0Normal':            "0Normal",
    #   'tcp'    :            "tcp"
    # },

    'BoT_IoT': { #map
        '0Normal': "0Normal",
        'Data_Exfiltration': "theft",
        'HTTP': "HTTP",
        'Keylogging': "theft",
        'OS_Fingerprint': "scan",
        'Service_Scan': "scan",
        'TCP': "TCP",
        'UDP': "UDP"
    },
    # ['udp' 'tcp' 'scan' 'syn' 'ack' 'udpplain' 'combo' 'junk' '0Normal']

    "UNSW_NB15":{ # not map nhưng bỏ 1 label Analysis 
        # '0Normal': "0Normal",
        # 'Exploits':'Malware',
        # 'Shellcode':'Malware',
        # 'Backdoor':'Malware',
        # 'Worms':'Malware',
        # 'DoS': 'DoS',
        # 'DoS': 'Fuzzers',
        # 'Reconnaissance':'Scan',
        # 'Analysis':'Scan'
        '0Normal': "0Normal",
        'Exploits':'Exploits',
        'Shellcode':'Shellcode',
        'Backdoor':'Backdoor',
        'Worms':'Worms',
        'DoS': 'DoS',
        'DoS': 'Fuzzers',
        'Reconnaissance':'Reconnaissance',
        'Analysis':'Analysis'
    },
    "ToN_IoT":{ # not map 
        'backdoor': 'Malware',
        'ransomware': 'Malware',
        'scanning': 'Scan',
        'password': 'BruteForce',
        'ddos': 'DDoS',
        'xss': 'WebAttack',
        'injection':'WebAttack',
        'dos': 'DoS',
        'mitm': 'MITM'
    }
}

def load_n_baiot():
    data_dir = os.path.join(__DATA_DIR,"N_BaIoT")
    df_nors, df_anos = Import_dataset(data_dir, devices = __DEVICES, bot_type = 2, atk_type = "ALL")
    
    # dropping duplicate values 
    # df_nors.drop_duplicates(keep=False, inplace=True) 
    # df_anos.drop_duplicates(keep=False, inplace=True) 
    
    df_anos = Get_sample_ratio_by_column(df_anos, "Names Atk", __LIMIT_CNT)
    df_nors = Get_sample_ratio_by_column(df_nors, "Label", __LIMIT_CNT)
   
    df_anos = Preprocess_data(df_anos)
    df_nors = Preprocess_data(df_nors)
    df =  pd.concat([df_anos,df_nors], ignore_index=True).reset_index(drop=True)
    return df


def load_iotid20():
    data_dir = os.path.join(__DATA_DIR,"IoTID20")
    df = pd.read_csv(f"{data_dir}/iotid20.csv")
    # dropped_cols =["Flow_ID","Src_IP","Dst_IP","Timestamp"]
    # df.drop(columns=dropped_cols, inplace=True)
    # df.drop_duplicates(inplace=True, ignore_index=True)
    
    df = Get_sample_ratio_by_column(df, "Target", __LIMIT_CNT)    
   
    # df.drop(columns=dropped_cols, inplace=True, errors='ignore')
    df.rename(columns={'Target': __TARGET}, inplace=True)
    
    return df

def load_ids_by_name(name):
    df = None
    if name == "IoTID20":
        df = load_iotid20()
    else:
        if name == "N_BaIoT":
            df = load_n_baiot()
        else:
            df = load_data_ids(name, __DATA_DIR, __LIMIT_CNT)
    # df['Datasets'] = np.full(df.shape[0], name)
    df = df.drop(columns = ['Binary_dtloader','Category_dtloader'], errors='ignore')
    df[__TARGET] = df[__TARGET].apply(lambda x: "0Normal" if (x in ['BENIGN', 'Normal','normal','Benign','BenignTraffic']) else x)
    
    # df = df[df[__TARGET].isin(__Atks_profile[name][__Target_atk] + ["0Normal"])]

    
    # Bỏ comment cái này
    if name != "N_BaIoT" and name != "IoTID20" and name != "ToN_IoT":
        keys_list = list(_category_map[name].keys())
        df  = df[df[__TARGET].isin(keys_list)]
        
        df[__TARGET] = df[__TARGET].apply(lambda x: _category_map[name][x])
    return df


## Function to Create synthetic Data

In [38]:
from numpy import random as np_random
import random

def generate_multiclass_dataset(N, M, normal_points_per_cluster, n_features, anomaly_points):

    # Generate clusters
    if __DATA_TYPE == "MIX CLUSTER":
        data, labels = make_blobs(n_samples=N * normal_points_per_cluster, n_features = n_features,
                              centers=N, cluster_std=10.0, random_state=__SEED)
    if  __DATA_TYPE == "CLUSTER":
        data, labels = make_blobs(n_samples=M * normal_points_per_cluster, n_features = n_features,
                              centers=M, cluster_std=10.0, random_state=__SEED)
    if __DATA_TYPE == "MIX":
        data, labels = make_multilabel_classification(n_samples=M * normal_points_per_cluster, 
                                        n_features = n_features, allow_unlabeled = False,
                                  n_classes=M, random_state=__SEED)
        labels = np.argmax(labels, axis=1)

    if __DATA_TYPE == "MIX CLUSTER":
        data, labels = merge_and_encode_labels(data, labels, M)
    

    return data, labels


def merge_and_encode_labels(data, labels, M):
    """
    Merges labels randomly until M unique labels are left and then encodes these labels.

    :param data: 2D numpy array where each row is a sample.
    :param labels: 1D numpy array of labels corresponding to the data samples.
    :param M: The number of unique labels to be left after merging.
    :return: Updated data and labels.
    """
    unique_labels = np.unique(labels)
    n_unique = len(unique_labels)

    if M >= n_unique:
        print("M is greater than or equal to the number of unique labels. No merging needed.")
        return data, labels

    # Randomly merge labels
    while n_unique > M:
        # Pick two different labels randomly
        label1, label2 = random.sample(list(unique_labels), 2)

        # Merge label2 into label1
        labels[labels == label2] = label1

        # Update unique labels
        unique_labels = np.unique(labels)
        n_unique = len(unique_labels)

    # Label encoding
    encoder = LabelEncoder()
    encoded_labels = encoder.fit_transform(labels)

    return data, encoded_labels

def generate_random_array(N, S):
    """
    Generate a random array of size N with sum equal to S.

    Parameters:
    - N: Size of the output array.
    - S: Sum of all values in the output array.

    Returns:
    - random_array: Generated random array.
    """
    # Generate N-1 random values between 0 and S
    random_values = np.random.uniform(0, S, N-1)
    
    # Sort the random values and insert 0 at the beginning and S at the end
    random_values = np.sort(random_values)
    random_values = np.insert(random_values, 0, 0)
    random_values = np.append(random_values, S)
    
    # Compute the differences between consecutive values to get the array
    random_array = np.diff(random_values)

    return random_array

# This def to create perfect synthetic data

def generate_independently_vectors(n_cls, n_fts, n_sample_train, n_sample_test):

    if n_fts < n_sample_train:
        print("n_fts > n_sample_train")
        return None

    
    n_ = np.array([n_sample_train // n_cls for i in range(n_cls)])
    m_ = np.array([n_sample_test // n_cls for i in range(n_cls)])
    n_[n_cls-1] += (n_sample_train - (n_sample_train // n_cls) * n_cls)
    m_[n_cls-1] += (n_sample_test - (n_sample_test // n_cls) * n_cls)

    # if n_fts < n_sample_train:
    #     n_add = np.array([n_fts // n_cls for i in range(n_cls)])
    #     n_add[n_cls-1] += (n_fts - (n_fts // n_cls) * n_cls)
    #     n_add = n_add - n_
    # else:
    #     n_add = np.zeros(n_sample_train)
    
    X_train = np.zeros((n_sample_train,n_fts))
    y_train = np.zeros(n_sample_train)

    X_test = np.zeros((n_sample_test,n_fts))
    y_test = np.zeros(n_sample_test)

    id = 0
    id_t = 0
    for k in range(n_cls):
        x_cls = np.zeros(n_[k])
        for i in range(n_[k]):
            X_train[id+i][id+i] = np_random.randint(1,3)
            y_train[id+i] = k
       

        for i in range(m_[k]):
            tmp = generate_random_array(n_[k],1)
            for t in range(n_[k]):
                X_test[id_t + i][id+t] = tmp[t] * X_train[id+t][id+t]
                # X_test[id_t + i][id+t] = tmp[t]
                # res_tmp = tmp[t]
                # for x in range(n_[k]):
                    # res_tmp = res_tmp + res_tmp*X_train[id+x][id+x]
                    
                # X_test[id_t + i][id+t] = res_tmp
                
            y_test[id_t+i] = k
            
        id = id + n_[k]
        id_t = id_t + m_[k]
        
    return X_train, X_test, y_train, y_test






## Set Logger

In [39]:
def setup_logger(log_file='example.log', level=logging.DEBUG):
    # Create a logger
    logger = logging.getLogger('my_logger')
    logger.setLevel(level)

    # Create a file handler and set the level to the specified level
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(level)

    # Create a formatter and add it to the file handler
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add the file handler to the logger
    logger.addHandler(file_handler)

    return logger




In [40]:
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.metrics import classification_report


import numpy as np
from sklearn.metrics.pairwise import pairwise_kernels


def K(X,Y=None,metric='poly',coef0=1,gamma=None,degree=3):
    if metric == 'poly':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'linear':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'sigmoid':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    elif metric == 'rbf':
        k = pairwise_kernels(X,Y=Y,metric=metric)
    return k

def kernel_distance_matrix(X=None, kernel="linear"):
    """
    Calculate the distance matrix using the kernel trick.

    Parameters:
    - X: Input data, a 2D numpy array where each row represents a sample.
    - kernel: Kernel function. Default is __KERNEL.

    Returns:
    - distance_matrix: Distance matrix.
    """
    # Calculate kernel matrix
    kernel_matrix = K(X, metric=kernel)

    # Calculate distance matrix using the given formula
    diagonal = np.diagonal(kernel_matrix)
    distance_matrix = np.outer(diagonal, np.ones(X.shape[0])) - 2 * kernel_matrix + np.outer(np.ones(X.shape[0]), diagonal)

    return distance_matrix

def kernel_distance(matrix1, matrix2, kernel="linear"):
    """
    Calculate the distance between two matrices using the kernel trick.

    Parameters:
    - matrix1: The first input matrix (NumPy array).
    - matrix2: The second input matrix (NumPy array).
    - gamma: The gamma parameter for the RBF kernel.

    Returns:
    - distance_matrix: The distance matrix between the two input matrices.
    """
    if matrix1.shape[1] != matrix2.shape[1]:
        raise ValueError("The number of features in the input matrices must be the same.")
        
    Kaa = []
    for i in range(len(matrix1)):
        Kaa.append(K(matrix1[i,:].reshape(1,-1),metric=kernel))    
    Kaa = np.asarray(Kaa).ravel().reshape(len(Kaa),1)
    
    Kab = K(matrix1,matrix2,metric=kernel)
    Kbb = []
    for i in range(len(matrix2)):
        Kbb.append(K(matrix2[i,:].reshape(1,-1),metric=kernel))
    Kbb = np.asarray(Kbb).ravel()
    
    d = Kaa-2*Kab+Kbb #shape: (matrix1,matrix2)

    return d

In [41]:
# a = np.array([[[1,2,3]],[[11,2,3]]])
# print(len(a.shape))


In [42]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def visualize_scaled_distribution(data):
    """
    Visualize the distribution of data after scaling it to a normal distribution using Z-score normalization.

    Parameters:
    - data: Input NumPy array.

    Returns:
    - None (displays the histogram plot).
    """
    # Apply Z-score normalization to scale the data to a normal distribution
    scaled_data = stats.zscore(data)

    # Create a histogram of the scaled data
    plt.hist(scaled_data, bins=20, color='skyblue', edgecolor='black')

    # Add labels and title
    plt.title('Distribution of Scaled Data (Normal Distribution)')
    plt.xlabel('Scaled Data Values')
    plt.ylabel('Frequency')

    # Display the plot
    plt.show()




# Create independent vectors


In [43]:
def preprocess_data(data, poly):
    datasets = data.to_numpy()
    labels = datasets[:,-1]
    dataset = datasets[:,:-1]
        
    logger.info("Distribution of Labels:")
    logger.info(np.unique(labels, return_counts=True))
    # print(type(dataset))
    print("Dataset size:")
    print(dataset.shape)

    ## ========================== Running Main Model ================================================
    if __DATASET == "IoTID20":
        # dataset[np.isinf(dataset)] = np.nan
        dataset[dataset == -np.inf] = np.nan
        dataset[dataset == np.inf] = np.nan
        mean_imputer_X = SimpleImputer(strategy="mean")
        dataset = mean_imputer_X.fit_transform(dataset)

    
    if poly > 0:
        from sklearn.preprocessing import PolynomialFeatures
        poly_model = PolynomialFeatures(poly,interaction_only=True)
        dataset = poly_model.fit_transform(dataset)
        print(type(dataset),dataset.shape)
        

    return dataset, labels, data.columns


def split_into_batches(arr, num_batches):
    batch_size = len(arr) // num_batches  # Integer division to find base batch size
    remainder = len(arr) % num_batches   # Remaining elements

    batches = []
    start_idx = 0
    for _ in range(num_batches):
        end_idx = start_idx + batch_size
        
        batches.append(arr[start_idx:end_idx])
        start_idx = end_idx
        
    if remainder > 0:
        batches.append(arr[end_idx:end_idx+remainder])
        
    return batches


In [44]:
# def gaussian_elimination(vectors, threshold=1e-10):
#         """Performs Gaussian elimination while preserving the order of vectors.
    
#         Args:
#             vectors: A numpy array where each row represents a vector.
#             threshold: A value below which numbers are considered zero for numerical stability.
    
#         Returns:
#             Two lists:
#                 - A list of linearly independent vectors in their original order.
#                 - A list of indices indicating the positions of the linearly independent vectors
#                 in the original input array.
#         """
    
#         matrix = vectors.copy()  # Ensure original data isn't modified
#         num_rows, num_cols = matrix.shape
#         independent_indices = []
#         max_rank = min(num_rows, num_cols)
    
#         for col in range(max_rank):
#             # Find the pivot row (the row with the largest absolute value in the current column)
#             pivot_row = np.argmax(np.abs(matrix[col:, col])) + col  # Vectorized operation
    
#             # If the pivot element is close to zero, skip this column
#             if abs(matrix[pivot_row, col]) < threshold:
#                 continue
    
#             # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
#             if pivot_row != col:
#                 matrix[[col, pivot_row]] = matrix[[pivot_row, col]]
    
#             # Normalize the pivot row and eliminate elements below
#             pivot = matrix[col, col]
#             matrix[col] /= pivot
    
#             # Eliminate the elements below the pivot (vectorized operation)
#             rows_below = matrix[col+1:, col]  
#             matrix[col+1:] -= np.outer(rows_below, matrix[col])
    
#             independent_indices.append(pivot_row)
    
#         # Extract linearly independent vectors (non-zero rows) in their original order
#         independent_vectors = vectors[independent_indices]  # Direct indexing
    
#         return independent_vectors, independent_indices


def gaussian_elimination(vectors, threshold=1e-10):
        """Performs Gaussian elimination while preserving the order of vectors.
    
        Args:
            vectors: A numpy array where each row represents a vector.
            threshold: A value below which numbers are considered zero for numerical stability.
    
        Returns:
            Two lists:
                - A list of linearly independent vectors in their original order.
                - A list of indices indicating the positions of the linearly independent vectors
                  in the original input array.
        """
    
        matrix = vectors.copy()
        num_rows, num_cols = matrix.shape
        independent_indices = []  # Track indices of independent vectors
    
        max_rank = min(num_rows, num_cols)
    
        row = 0
        for col in range(max_rank):
            # Find the pivot row (the row with the largest absolute value in the current column)
            pivot_row = row
            for i in range(row + 1, num_rows):
                if abs(matrix[i, col]) > abs(matrix[pivot_row, col]):
                    pivot_row = i
    
            # If the pivot element is close to zero, skip this column (the vector is linearly dependent)
            if abs(matrix[pivot_row, col]) < threshold:
                continue
    
            # Swap the current row with the pivot row ONLY IF THEY ARE DIFFERENT
            if pivot_row != row:
                matrix[[row, pivot_row]] = matrix[[pivot_row, row]]
    
            # Normalize the pivot row (make the pivot element equal to 1)
            pivot = matrix[row, col]
            matrix[row] /= pivot
    
            # Eliminate the elements below the pivot
            for i in range(row + 1, num_rows):
                factor = matrix[i, col]
                matrix[i] -= factor * matrix[row]
    
            independent_indices.append(pivot_row)  # Store the original index
            row += 1
    
        # Extract linearly independent vectors (non-zero rows) in their original order
        independent_vectors = [vectors[i] for i in independent_indices]
    
        return independent_vectors, independent_indices


def form_independent(X_train, y_train, kernel):
    dataX = []
    datay = []
    labels = np.unique(y_train)
    # print("Debug ===== labels:", labels)
    for label in labels:
        mask = np.array([y == label for y in y_train])
        data = X_train[mask]
        if kernel is not None:
            data = metrics.pairwise_kernels(data, metric=kernel)
            data = np.array(data)
        _, indicates = gaussian_elimination(data)
        # print(type(data), data.shape)
        dataX = dataX + X_train[mask][indicates].tolist()
        datay = datay + [label]*len(indicates)
    # print("Debug ===== dataX:", np.array(dataX).shape)
    return np.array(dataX), np.array(datay)


def form_independent_bybatch(X_train, y_train, batch, kernel):
    dataX = []
    datay = []
    labels = np.unique(y_train)
    # print("Debug ===== labels:", labels)
    for label in labels:
        
        mask = np.array([y == label for y in y_train])
        data = X_train[mask]
        datas = split_into_batches(data,batch)
        tmp_data = np.array([[0]*X_train.shape[1]])
        
        for x in datas:
            
            tmp_data = np.concatenate((tmp_data, x), axis =0)
            # print("Debug:", tmp_data.shape)
            if kernel is not None:
                tmp = metrics.pairwise_kernels(tmp_data, data, metric=kernel)
                # print("Debug ===== tmp_data:", tmp_data.shape)
            
                _, indicates = gaussian_elimination(tmp)
                tmp_data = tmp_data[indicates]
            else:
                _, indicates = gaussian_elimination(tmp_data)
                tmp_data = tmp_data[indicates]
            
        dataX = dataX + tmp_data.tolist()
        datay = datay + [label]*len(tmp_data)
    # print("Debug ===== dataX:", np.array(dataX).shape)
    return np.array(dataX), np.array(datay)

In [45]:
## Load Data block

__SEED = __DEFAULT_RANDOM_SEED
seedEverything(__DEFAULT_RANDOM_SEED)

__PREFIX_DIR = "/home/jupyter-hanx"
__WORKING_DIR = f"{__PREFIX_DIR}/HHH"
__DATASET = "IoTID20"
__DATASETS = ["BoT_IoT","ToN_IoT","N_BaIoT","UNSW_NB15",'CIC_IoT2023','IoTID20']
__DATA_DIR = os.path.join(__PREFIX_DIR,'Datasets')
__DATA_DIR_OUT = os.path.join(__WORKING_DIR,'Data')
__LIMIT_CNT = 5000
__POLY = 2
__TARGET = "Label"
__MODE = "Unsup" # "Novel"
__DATA_TYPE = __DATASET


logger = setup_logger(level=logging.WARNING)



In [46]:
df = load_ids_by_name(__DATASET)

print(df[__TARGET].unique())

print(df[__TARGET].value_counts())

df.head(5)

['Mirai' 'DoS' '0Normal' 'Scan' 'MITM ARP Spoofing']
Label
Mirai                5000
DoS                  5000
0Normal              5000
Scan                 5000
MITM ARP Spoofing    5000
Name: count, dtype: int64


,Src_Port,Dst_Port,Protocol,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,TotLen_Fwd_Pkts,TotLen_Bwd_Pkts,Fwd_Pkt_Len_Max,Fwd_Pkt_Len_Min,...,Fwd_Seg_Size_Min,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label
0,60424,443,6,130,1,1,261.0,0.0,261.0,261.0,...,0,0.0,0.0,0.0,0.0,130.0,0.000000,130.0,130.0,Mirai
1,52976,9010,6,80,1,1,480.0,0.0,480.0,480.0,...,0,0.0,0.0,0.0,0.0,80.0,0.000000,80.0,80.0,Mirai
2,56204,9020,6,157,0,3,0.0,4164.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,78.5,4.949747,82.0,75.0,Mirai
3,52930,9020,6,150,2,1,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,75.0,1.414214,76.0,74.0,Mirai
4,9020,56211,6,150,0,3,0.0,0.0,0.0,0.0,...,0,0.0,0.0,0.0,0.0,75.0,5.656854,79.0,71.0,Mirai


In [47]:
df.shape

(25000, 80)

In [48]:
def process_combination(args):
    """Function to process a single combination of poly and kernel."""
    poly, kernel = args
    # global __POLY, __kernel
    # __POLY = poly  # Set the global variable for the subprocess
    # __kernel = kernel    
    data, label, columns_name = preprocess_data(df.copy(), poly) 
    data_iv, label_iv = form_independent(data.copy(), label.copy(), kernel)
    tmp = np.concatenate((data_iv, label_iv.reshape(-1, 1)), axis=1)
    
    print(f"Data shape of poly {poly} and kernel {str(kernel)}:", tmp.shape)

    output_path = os.path.join(__DATA_DIR_OUT, __DATASET, str(__LIMIT_CNT), f"poly{poly}_kernel{str(kernel)}.csv")
    pd.DataFrame(tmp).to_csv(output_path)

In [49]:
from multiprocessing import Pool  # Import Pool for process management

# for poly in [0,2,3]:
#     __POLY = poly
#     for kernel in __kernel:
#         data, label, columns_name = preprocess_data(df.copy())
#         data_iv, label_iv = form_independent(data.copy(), label.copy(), kernel)
#         tmp = np.concatenate((data_iv, label_iv.reshape(-1,1)), axis=1)
#         print("Prepare dataset:", f'poly{__POLY}_kernel{str(kernel)}.csv')
#         print("Data shape:", tmp.shape)
#         pd.DataFrame(tmp).to_csv(os.path.join(__DATA_DIR_OUT,__DATASET,f'poly{__POLY}_kernel{str(kernel)}.csv'))
output_path = os.path.join(__DATA_DIR_OUT,  __DATASET, str(__LIMIT_CNT),f"{__LIMIT_CNT}.csv")
df.to_csv(output_path)
_polys = [0,2]
# ... (Your __kernel list)
__kernels = [None,'rbf','poly','linear','sigmoid']
# Create combinations of poly and kernel
combinations = [(poly, kernel) for poly in _polys for kernel in __kernels]

# Use multiprocessing pool
with Pool() as pool:  # Automatically manages processes
    pool.map(process_combination, combinations)  # Parallelize the work


Dataset size:Dataset size:Dataset size:Dataset size:



(25000, 79)Dataset size:(25000, 79)(25000, 79)(25000, 79)




(25000, 79)
Dataset size:
(25000, 79)
Dataset size:
(25000, 79)
Dataset size:
(25000, 79)
Dataset size:Dataset size:

(25000, 79)(25000, 79)

<class 'numpy.ndarray'> (25000, 3161)
<class 'numpy.ndarray'> (25000, 3161)
<class 'numpy.ndarray'> (25000, 3161)
<class 'numpy.ndarray'> (25000, 3161)
<class 'numpy.ndarray'> (25000, 3161)
Data shape of poly 0 and kernel None: (258, 80)
Data shape of poly 0 and kernel sigmoid: (5, 80)
Data shape of poly 2 and kernel sigmoid: (5, 3162)


KeyboardInterrupt: 